In [27]:
import torch
import torch.nn as nn

In [28]:
text = """
I love Python
I love programming
Python is powerful
Python is easy
I love machine learning
Python programming is fun
machine learning is interesting
I enjoy Python programming
"""
token=text.lower().split()
print(token)

['i', 'love', 'python', 'i', 'love', 'programming', 'python', 'is', 'powerful', 'python', 'is', 'easy', 'i', 'love', 'machine', 'learning', 'python', 'programming', 'is', 'fun', 'machine', 'learning', 'is', 'interesting', 'i', 'enjoy', 'python', 'programming']


In [29]:
vocabulary=sorted(set(token))
print(vocabulary)

['easy', 'enjoy', 'fun', 'i', 'interesting', 'is', 'learning', 'love', 'machine', 'powerful', 'programming', 'python']


In [30]:
wordtoid={
    word:id for id,word in enumerate(vocabulary)
}
idtoword={
    id:word for word,id in wordtoid.items()
}
print("Word To Id:",wordtoid)
print("Id to Word:",idtoword)

Word To Id: {'easy': 0, 'enjoy': 1, 'fun': 2, 'i': 3, 'interesting': 4, 'is': 5, 'learning': 6, 'love': 7, 'machine': 8, 'powerful': 9, 'programming': 10, 'python': 11}
Id to Word: {0: 'easy', 1: 'enjoy', 2: 'fun', 3: 'i', 4: 'interesting', 5: 'is', 6: 'learning', 7: 'love', 8: 'machine', 9: 'powerful', 10: 'programming', 11: 'python'}


In [31]:
tokenids=[wordtoid[i] for i in token]
print("Token IDS:",tokenids)

Token IDS: [3, 7, 11, 3, 7, 10, 11, 5, 9, 11, 5, 0, 3, 7, 8, 6, 11, 10, 5, 2, 8, 6, 5, 4, 3, 1, 11, 10]


In [32]:
input_ids=tokenids[:-1]
target_ids=tokenids[1:]
print("Input ids:",input_ids)
print("Target ids:",target_ids)

Input ids: [3, 7, 11, 3, 7, 10, 11, 5, 9, 11, 5, 0, 3, 7, 8, 6, 11, 10, 5, 2, 8, 6, 5, 4, 3, 1, 11]
Target ids: [7, 11, 3, 7, 10, 11, 5, 9, 11, 5, 0, 3, 7, 8, 6, 11, 10, 5, 2, 8, 6, 5, 4, 3, 1, 11, 10]


In [33]:
# convert to tensors
input_tensor=torch.tensor([input_ids],dtype=torch.long)
target_tensor=torch.tensor([target_ids],dtype=torch.long)
print("Input Tensor:",input_tensor)
print("Input Shape:",input_tensor.shape)
print("Target Tensor:",target_tensor)
print("Target Shape:",target_tensor.shape,)

Input Tensor: tensor([[ 3,  7, 11,  3,  7, 10, 11,  5,  9, 11,  5,  0,  3,  7,  8,  6, 11, 10,
          5,  2,  8,  6,  5,  4,  3,  1, 11]])
Input Shape: torch.Size([1, 27])
Target Tensor: tensor([[ 7, 11,  3,  7, 10, 11,  5,  9, 11,  5,  0,  3,  7,  8,  6, 11, 10,  5,
          2,  8,  6,  5,  4,  3,  1, 11, 10]])
Target Shape: torch.Size([1, 27])


In [34]:
# LSTM Model
class LSTM(nn.Module):
    def __init__(self,vocabularysize,embeddingdim,hiddensize):
        super().__init__()
        self.embedding=nn.Embedding(vocabularysize,embeddingdim)
        self.lstm=nn.LSTM(input_size=embeddingdim,hidden_size=hiddensize,batch_first=True)
        self.linear=nn.Linear(hiddensize,vocabularysize)

    def forward(self,x):
        Embeddings=self.embedding(x)
        output,(hidden,cell)=self.lstm(Embeddings)
        output=self.linear(output)
        return output

In [35]:
vocabularysize=len(vocabulary)
embeddingdim=10
hiddensize=20
model=LSTM(vocabularysize,embeddingdim,hiddensize)
lossfunction=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(model.parameters(),lr=0.01)

In [36]:
for epoch in range(1000):
    output=model(input_tensor)
    output=output.reshape(-1,vocabularysize)
    target=target_tensor.reshape(-1)
    loss=lossfunction(output,target)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if epoch % 100 == 0:
        print("Epoch:",epoch,"Loss:",loss.item())

Epoch: 0 Loss: 2.4422366619110107
Epoch: 100 Loss: 0.014255432412028313
Epoch: 200 Loss: 0.004717856179922819
Epoch: 300 Loss: 0.0024661582428961992
Epoch: 400 Loss: 0.0015414906665682793
Epoch: 500 Loss: 0.0010848173405975103
Epoch: 600 Loss: 0.0008145316387526691
Epoch: 700 Loss: 0.0006385876913554966
Epoch: 800 Loss: 0.0005163333262316883
Epoch: 900 Loss: 0.00042717583710327744


In [ ]:
model.eval()
with torch.no_grad():
    output=model(input_tensor)
    print(output.shape)
    prediction_ids=torch.argmax(output,dim=1) # Predict every position

print("Prediction IDS:",prediction_ids)

predicted_words = []
for predicted_id in prediction_ids:

    word = idtoword[
        predicted_id.item()
    ]

    predicted_words.append(word)


print("Predicted Words:", predicted_words)

torch.Size([9, 12])
Prediction IDS: tensor([ 7, 11,  3,  7, 10, 11,  5,  9, 11])
Predicted Words: ['love', 'python', 'i', 'love', 'programming', 'python', 'is', 'powerful', 'python']


In [63]:
text="I love python i love programming python is powerful python is easy i love"
tokens=text.lower().split()
texttoid=[wordtoid[i] for i in tokens]
input_tensor=torch.tensor(texttoid,dtype=torch.long)

print(input_tensor.shape)
with torch.no_grad():

    output = model(input_tensor)
    print(output.shape)

    # Take the output of the LAST word
    # last_output = output[:, -1, :] # if batch
    last_output = output[-1, :] # not batch

    # Find the word with highest score
    predicted_id = torch.argmax(
        last_output,
        dim=-1
    )

predicted_word = idtoword[
    predicted_id.item()
]
print("Next word:",text, predicted_word)

torch.Size([14])
torch.Size([14, 12])
Next word: I love python i love programming python is powerful python is easy i love machine


In [80]:
text="i"
length=118
final=[]
for i in range(length):
    tokens=text.lower().split()
    texttoid=[wordtoid[i] for i in tokens]
    input_tensor=torch.tensor(texttoid,dtype=torch.long)
    # print(input_tensor.shape)
    with torch.no_grad():
        output = model(input_tensor)
        # print(output.shape)
        last_output = output[-1, :]
        predicted_id = torch.argmax(last_output,dim=-1)
    predicted_word = idtoword[predicted_id.item()]
    final.append(predicted_word)
    text=text+" "+predicted_word
    length+=1
print(text)
    

i love python i love programming python is powerful python is easy i love machine learning python programming is fun machine learning is interesting i enjoy python programming is fun machine learning is interesting i enjoy python programming is fun machine learning is interesting i enjoy python programming is fun machine learning is interesting i enjoy python programming is fun machine learning is interesting i enjoy python programming is fun machine learning is interesting i enjoy python programming is fun machine learning is interesting i enjoy python programming is fun machine learning is interesting i enjoy python programming is fun machine learning is interesting i enjoy python programming is fun machine learning is interesting i enjoy python programming is


In [ ]:
# NEXT ATTENTION